# 11. 잔차 보정 (2단계 모델) · 못 맞춘 영화를 역으로 이용하기

**1단계**: 09 TabPFN + KOBIS (Public 436,672, 현재 최고)
**2단계**: 1단계가 **얼마나 틀렸는지(잔차)**를 예측하는 모델 → 최종 = 1단계 예측 + 보정

| 단계 | 내용 |
| --- | --- |
| 1 | 1단계 잔차 확인: 어떤 영화를 어느 방향으로 틀리는지 |
| 2 | 2단계 모델 × 잔차 종류 비교 (CV) |
| 3 | 보정 강도(α) 비교: 최종 = 1단계 + α × 보정 |
| 4 | 제출 파일 |

**누수 방지**
- 잔차는 반드시 1단계의 **OOF 예측**(그 영화를 학습에 안 쓴 모델의 예측)으로 계산
- 2단계 평가는 **다른 seed의 5-Fold**로 한 번 더 나눠서 OOF로 계산 (스태킹 방식)
- 보정 강도 α를 같은 CV로 고르면 낙관적일 수 있음 → α=0.5 같은 약한 보정 파일도 같이 만들기

## 0. 설정 · 1단계 결과 불러오기

In [1]:
import os
import re
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
train = pd.read_csv('movies_train.csv')
test = pd.read_csv('movies_test.csv')
submission = pd.read_csv('submission.csv')
kobis = pd.read_csv('kobis_movies.csv')
y = train['box_off_num'].values.astype(float)
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = 'cache_11'
os.makedirs(CACHE_DIR, exist_ok=True)

stage1 = pickle.load(open('cache_09/tabpfn.pkl', 'rb'))
p1_oof, p1_test = stage1['oof'], stage1['test']
print(f'1단계 (09 TabPFN + KOBIS) CV RMSE {rmse(y, p1_oof):,.0f} | 흥행작 {rmse(y[hit], p1_oof[hit]):,.0f}')

1단계 (09 TabPFN + KOBIS) CV RMSE 363,107 | 흥행작 837,828


C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 09와 같은 변수 (06 최종 + KOBIS)
norm = lambda s: re.sub(r'[^0-9a-zA-Z가-힣]', '', str(s)).lower()
kobis['key'] = kobis['movieNm'].map(norm)
kobis['open_dt'] = pd.to_datetime(kobis['openDt'], errors='coerce')
K_COLS = ['open_scrn', 'open_show', 'max_scrn', 'max_show', 'days_top10', 'best_rank']


def kobis_features(df):
    rows = []
    for title, r in zip(df['title'], pd.to_datetime(df['release_time'])):
        cand = kobis[kobis['key'] == norm(title)]
        row = {f'k_{c}': np.nan for c in K_COLS}
        if len(cand):
            gap = (cand['open_dt'] - r).abs().dt.days.fillna(9999)
            if gap.min() <= 30:
                best = cand.loc[gap.idxmin()]
                row = {f'k_{c}': best[c] for c in K_COLS}
        rows.append(row)
    K = pd.DataFrame(rows)
    K['k_in_top10'] = K['k_max_scrn'].notna().astype(int)
    K['k_best_rank'] = K['k_best_rank'].fillna(11)
    return K.fillna(0).astype(float)


X, X_test = make_features(train, test, {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1})
X = pd.concat([X.reset_index(drop=True), kobis_features(train)], axis=1)
X_test = pd.concat([X_test.reset_index(drop=True), kobis_features(test)], axis=1)

# 2단계 입력 = 원래 변수 + 1단계 예측 (sqrt 단위)
X2 = X.assign(p1_sqrt=np.sqrt(p1_oof))
X2_test = X_test.assign(p1_sqrt=np.sqrt(p1_test))
print('2단계 변수', X2.shape[1], '개')

2단계 변수 49 개


## 1. 1단계 잔차 확인
잔차 = 실제 − 예측 (양수 = 과소 예측, 음수 = 과대 예측)

In [3]:
res = y - p1_oof
r = pd.DataFrame({'제목': train['title'], '실제': y, '1단계 예측': p1_oof, '잔차': res,
                  '개봉일 스크린': X['k_open_scrn'], '최대 스크린': X['k_max_scrn'], '상위10 일수': X['k_days_top10']})
print('제곱오차 중 상위 10편 비중: %.0f%%' % (np.sort(res**2)[::-1][:10].sum() / (res**2).sum() * 100))
print('\n[과소 예측 상위 8편]')
display(r.sort_values('잔차', ascending=False).head(8).style.format({c: '{:,.0f}' for c in r.columns if c != '제목'}))
print('[과대 예측 상위 8편]')
display(r.sort_values('잔차').head(8).style.format({c: '{:,.0f}' for c in r.columns if c != '제목'}))

# 잔차가 어떤 변수와 관련 있는지 (2단계가 배울 수 있는 신호가 있는지)
corr = X2.corrwith(pd.Series(res), method='spearman').sort_values(key=abs, ascending=False).head(10)
display(corr.to_frame('잔차와의 Spearman').style.format('{:.3f}'))

제곱오차 중 상위 10편 비중: 57%

[과소 예측 상위 8편]


,제목,실제,1단계 예측,잔차,개봉일 스크린,최대 스크린,상위10 일수
122,변호인,"11,374,879","8,495,714","2,879,165",591,925,60
288,타워,"5,181,014","3,023,113","2,157,901",597,638,38
372,국제시장,"14,262,766","12,194,612","2,068,154",931,"1,044",82
101,관상,"9,135,806","7,206,827","1,928,979",927,"1,240",46
530,7번방의 선물,"12,811,435","10,894,412","1,917,023",605,866,82
362,도둑들,"12,983,841","11,228,738","1,755,103",890,"1,091",68
561,숨바꼭질,"5,604,106","3,958,599","1,645,507",623,779,36
178,조선명탐정 : 사라진 놉의 딸,"3,872,015","2,471,067","1,400,948",740,825,29


[과대 예측 상위 8편]


,제목,실제,1단계 예측,잔차,개봉일 스크린,최대 스크린,상위10 일수
476,수상한 그녀,"8,659,725","11,436,890","-2,777,165",577,"1,027",71
472,감시자들,"5,508,299","7,421,600","-1,913,301",731,949,45
460,"광해, 왕이 된 남자","12,323,595","14,066,666","-1,743,071",688,"1,001",98
422,전설의 주먹,"1,744,585","3,467,440","-1,722,855",625,724,40
496,설국열차,"9,350,351","10,696,749","-1,346,398",936,"1,128",59
377,내부자들: 디 오리지널,"2,084,844","3,402,475","-1,317,631",595,838,35
486,뷰티 인사이드,"2,057,896","3,303,742","-1,245,846",633,648,41
434,미스터 고,"1,329,061","2,492,677","-1,163,616",840,840,26


,잔차와의 Spearman
p1_sqrt,-0.165
k_max_scrn,-0.134
k_open_scrn,-0.133
k_max_show,-0.131
k_days_top10,-0.129
k_open_show,-0.126
k_in_top10,-0.125
k_best_rank,0.124
num_staff,-0.121
is_big4,-0.103


## 2. 2단계 모델 비교
| 잔차 종류 | 2단계가 예측하는 것 | 최종 예측 |
| --- | --- | --- |
| `diff` | 실제 − 예측 (관객수 단위) | p1 + α × 보정 |
| `ratio` | log1p(실제) − log1p(예측) (몇 배 틀렸는지) | p1 × exp(α × 보정) |

| 2단계 모델 | 특징 |
| --- | --- |
| Ridge | 가장 단순 · 과적합 적음 |
| LightGBM (얕게) | 비선형 · 강한 규제 |
| TabPFN | 작은 데이터에 강함 |

In [4]:
kf2 = KFold(n_splits=5, shuffle=True, random_state=2024)   # 1단계와 다른 분할


def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 30)))


STAGE2 = {
    'Ridge':  make_ridge,
    'LGBM':   lambda: LGBMRegressor(n_estimators=300, learning_rate=0.02, num_leaves=7, min_child_samples=20,
                                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0,
                                    random_state=42, verbose=-1),
    'TabPFN': lambda: TabPFNRegressor.create_default_for_version(ModelVersion.V3_5, device=DEVICE, random_state=42),
}
RES_T = {
    'diff':  (lambda yy, p: yy - p,                    lambda p, c, a: p + a * c),
    'ratio': (lambda yy, p: np.log1p(yy) - np.log1p(p), lambda p, c, a: np.expm1(np.log1p(p) + a * c)),
}
Xr = X2.fillna(0)        # Ridge · LGBM용 (TabPFN은 결측 그대로)
Xr_test = X2_test.fillna(0)


def run_stage2(model, rtype):
    path = os.path.join(CACHE_DIR, f'{model}_{rtype}.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    make_r, _ = RES_T[rtype]
    target = make_r(y, p1_oof)
    A, A_test = (X2, X2_test) if model == 'TabPFN' else (Xr, Xr_test)
    corr_oof = np.zeros(len(y))
    t0 = time.time()
    for tr_idx, va_idx in kf2.split(A):
        m = STAGE2[model](); m.fit(A.iloc[tr_idx], target[tr_idx])
        corr_oof[va_idx] = m.predict(A.iloc[va_idx])
    m = STAGE2[model](); m.fit(A, target)
    out = {'corr_oof': corr_oof, 'corr_test': m.predict(A_test), '소요(초)': time.time() - t0}
    pickle.dump(out, open(path, 'wb'))
    return out


ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
rows, S2 = [], {}
for model in STAGE2:
    for rtype in RES_T:
        o = run_stage2(model, rtype)
        S2[(model, rtype)] = o
        _, apply = RES_T[rtype]
        for a in ALPHAS:
            f = np.clip(apply(p1_oof, o['corr_oof'], a), 0, None)
            rows.append({'2단계': model, '잔차': rtype, 'α': a, 'CV RMSE': rmse(y, f), '흥행작 RMSE': rmse(y[hit], f[hit])})
        print(f"{model:<7}{rtype:<6} α=1 CV {rows[-1]['CV RMSE']:>9,.0f} | α=0.5 CV {rows[-3]['CV RMSE']:>9,.0f} | {o['소요(초)']:.0f}초")

tbl = pd.DataFrame(rows)
base = rmse(y, p1_oof)
tbl['1단계 대비'] = tbl['CV RMSE'] - base

Ridge  diff   α=1 CV   364,853 | α=0.5 CV   363,935 | 0초
Ridge  ratio  α=1 CV   386,142 | α=0.5 CV   368,821 | 0초


LGBM   diff   α=1 CV   392,169 | α=0.5 CV   373,821 | 0초


LGBM   ratio  α=1 CV   395,492 | α=0.5 CV   371,853 | 0초


TabPFN diff   α=1 CV   375,314 | α=0.5 CV   368,883 | 81초


TabPFN ratio  α=1 CV   361,488 | α=0.5 CV   361,052 | 104초


## 3. 결과 (α = 보정 강도, 0이면 1단계 그대로)

In [5]:
piv = tbl.pivot_table(index=['2단계', '잔차'], columns='α', values='CV RMSE')
display(piv.style.format('{:,.0f}').highlight_min(axis=None, color='#c6efce'))
best = tbl.loc[tbl['CV RMSE'].idxmin()]
print(f"1단계 그대로: {base:,.0f}")
print(f"최고: 2단계 {best['2단계']} · 잔차 {best['잔차']} · α={best['α']} → CV {best['CV RMSE']:,.0f} ({best['1단계 대비']:+,.0f})")

1단계 그대로: 363,107
최고: 2단계 TabPFN · 잔차 ratio · α=0.75 → CV 360,967 (-2,140)


In [6]:
# 보정 전후: 1단계 오차가 컸던 영화들
key = (best['2단계'], best['잔차'])
_, apply = RES_T[best['잔차']]
f_best = np.clip(apply(p1_oof, S2[key]['corr_oof'], best['α']), 0, None)
idx = np.argsort(-(y - p1_oof) ** 2)[:10]
cmp = pd.DataFrame({'제목': train['title'].values[idx], '실제': y[idx], '1단계': p1_oof[idx], '보정 후': f_best[idx]})
cmp['좋아짐'] = np.where(np.abs(cmp['실제'] - cmp['보정 후']) < np.abs(cmp['실제'] - cmp['1단계']), '✅', '❌')
display(cmp.style.format({c: '{:,.0f}' for c in ['실제', '1단계', '보정 후']}))

,제목,실제,1단계,보정 후,좋아짐
0,변호인,"11,374,879","8,495,714","8,352,141",❌
1,수상한 그녀,"8,659,725","11,436,890","11,307,732",✅
2,타워,"5,181,014","3,023,113","2,988,721",❌
3,국제시장,"14,262,766","12,194,612","11,898,181",❌
4,관상,"9,135,806","7,206,827","7,219,843",✅
5,7번방의 선물,"12,811,435","10,894,412","10,729,550",❌
6,감시자들,"5,508,299","7,421,600","7,266,997",✅
7,도둑들,"12,983,841","11,228,738","11,043,558",❌
8,"광해, 왕이 된 남자","12,323,595","14,066,666","13,557,102",✅
9,전설의 주먹,"1,744,585","3,467,440","3,335,805",✅


## 4. 제출 파일
- CV 최고 조합 · CV에서 고른 α
- 같은 조합 · **α=0.5 (약한 보정)**: α를 CV로 고르면 낙관적이라 리더보드 확인용
- 개선이 1단계 대비 1% 미만이면 보정 효과가 오차 수준 → 09 유지 권장

In [7]:
def save(pred, fname, desc):
    sub = submission.copy(); sub['box_off_num'] = np.clip(pred, 0, None)
    sub.to_csv(fname, index=False)
    print(f'{fname:<34} {desc} | 예측 평균 {sub.box_off_num.mean():,.0f}')


tag = f"{best['2단계']}_{best['잔차']}"
if best['α'] > 0:
    save(apply(p1_test, S2[key]['corr_test'], best['α']), f"11_resid_{tag}_a{int(best['α']*100)}.csv",
         f"α={best['α']} · CV {best['CV RMSE']:,.0f}")
    half = tbl[(tbl['2단계'] == best['2단계']) & (tbl['잔차'] == best['잔차']) & (tbl['α'] == 0.5)]['CV RMSE'].iloc[0]
    save(apply(p1_test, S2[key]['corr_test'], 0.5), f'11_resid_{tag}_a50.csv', f'α=0.5 · CV {half:,.0f}')
print(f"\n개선폭: {-best['1단계 대비']:,.0f} ({-best['1단계 대비']/base*100:.1f}%)",
      '→ 1% 이상: 리더보드로 확인해 볼 만함' if -best['1단계 대비'] / base >= 0.01 else '→ 1% 미만: 오차 수준, 09 유지 권장')
tbl.to_csv('11_results.csv', index=False, encoding='utf-8-sig')

11_resid_TabPFN_ratio_a75.csv      α=0.75 · CV 360,967 | 예측 평균 648,179
11_resid_TabPFN_ratio_a50.csv      α=0.5 · CV 361,052 | 예측 평균 652,144

개선폭: 2,140 (0.6%) → 1% 미만: 오차 수준, 09 유지 권장
